# 🏗️ Parte 4 — Montando um bloco Transformer completo

Nas partes anteriores construímos a **atenção** do zero. Mas um Transformer tem mais algumas peças:

1. **Codificação posicional** → para saber a **ordem** das palavras;
2. **Múltiplas cabeças de atenção** (*multi-head*) → para olhar a frase de **vários jeitos**;
3. **Conexão residual + normalização** → para não "perder" a informação original;
4. **Rede feed-forward** → para "pensar" sobre cada palavra;
5. **Máscara** e **previsão da próxima palavra** → como o ChatGPT gera texto.

📌 Execute as células na ordem com `Shift + Enter`.

In [ ]:
import math

#                [ser vivo, comida, ação]
embeddings = {
    "o":         [0.1, 0.1, 0.1],
    "gato":      [0.9, 0.1, 0.0],
    "cachorro":  [0.8, 0.1, 0.1],
    "homem":     [0.9, 0.0, 0.1],
    "leite":     [0.0, 0.9, 0.1],
    "queijo":    [0.1, 0.8, 0.0],
    "bebeu":     [0.1, 0.3, 0.9],
    "comeu":     [0.1, 0.4, 0.8],
    "mordeu":    [0.1, 0.2, 0.9],
}

def produto_escalar(vetor_a, vetor_b):
    soma = 0
    for i in range(len(vetor_a)):
        soma = soma + vetor_a[i] * vetor_b[i]
    return round(soma, 2)

def softmax(lista_de_escores):
    exponenciais = []
    for valor in lista_de_escores:
        exponenciais.append(math.exp(valor))
    total = 0
    for valor in exponenciais:
        total = total + valor
    pesos = []
    for valor in exponenciais:
        pesos.append(round(valor / total, 3))
    return pesos

def mostrar_barras(palavras, pesos):
    for i in range(len(palavras)):
        porcentagem = pesos[i] * 100
        barra = "█" * int(porcentagem / 2)
        print("   ", palavras[i].ljust(10), barra, round(porcentagem, 1), "%")

def multiplicar_matriz_vetor(matriz, vetor):
    resultado = []
    for linha in matriz:
        resultado.append(produto_escalar(linha, vetor))
    return resultado

# Atenção da Parte 3, agora recebendo uma LISTA de vetores (um por posição da frase)
def atencao(vetores, W_Q, W_K, W_V, mascara=False):
    d = len(vetores[0])
    Q = []
    K = []
    V = []
    for vetor in vetores:
        Q.append(multiplicar_matriz_vetor(W_Q, vetor))
        K.append(multiplicar_matriz_vetor(W_K, vetor))
        V.append(multiplicar_matriz_vetor(W_V, vetor))

    saidas = []
    todos_os_pesos = []
    for i in range(len(vetores)):
        escores = []
        for j in range(len(vetores)):
            if mascara and j > i:
                escores.append(-1000)   # "proibido olhar o futuro" (exp(-1000) = 0)
            else:
                escores.append(round(produto_escalar(Q[i], K[j]) / math.sqrt(d), 2))
        pesos = softmax(escores)
        todos_os_pesos.append(pesos)

        saida = []
        for posicao in range(len(V[0])):
            soma = 0
            for j in range(len(vetores)):
                soma = soma + pesos[j] * V[j][posicao]
            saida.append(round(soma, 2))
        saidas.append(saida)
    return saidas, todos_os_pesos

W_Q = [[0, 0, 3],
       [0, 0, 3],
       [3, 3, 0]]
IDENTIDADE = [[1, 0, 0],
              [0, 1, 0],
              [0, 0, 1]]

print("Tudo pronto!")

## Passo 1 — O problema da ordem 🐕👨

Compare as frases:
- *"cachorro mordeu homem"* → normal;
- *"homem mordeu cachorro"* → notícia de jornal! 📰

As palavras são **as mesmas**, só a **ordem** muda. Será que a nossa atenção percebe a diferença?

In [ ]:
frase_1 = ["cachorro", "mordeu", "homem"]
frase_2 = ["homem", "mordeu", "cachorro"]

def frase_para_vetores(frase):
    vetores = []
    for palavra in frase:
        vetores.append(embeddings[palavra])
    return vetores

saidas_1, pesos_1 = atencao(frase_para_vetores(frase_1), W_Q, IDENTIDADE, IDENTIDADE)
saidas_2, pesos_2 = atencao(frase_para_vetores(frase_2), W_Q, IDENTIDADE, IDENTIDADE)

print("Frase 1:", frase_1)
for i in range(3):
    print("   ", frase_1[i].ljust(9), "->", saidas_1[i])
print("Frase 2:", frase_2)
for i in range(3):
    print("   ", frase_2[i].ljust(9), "->", saidas_2[i])

😮 O vetor de **"cachorro"** é **idêntico** nas duas frases! Para a atenção, a frase é só um "saco de palavras": ela **não sabe quem veio antes**.

## Passo 2 — Codificação posicional 📍

A solução é **somar ao embedding um vetor que representa a posição** (0, 1, 2, ...). Assim, "cachorro na posição 0" fica diferente de "cachorro na posição 2".

O artigo original usa **ondas** (seno e cosseno) com velocidades diferentes. Fazemos uma versão simplificada:

In [ ]:
def vetor_posicao(posicao, tamanho):
    vetor = []
    for i in range(tamanho):
        if i % 2 == 0:
            valor = math.sin(posicao / (2 ** i))   # posições pares: seno
        else:
            valor = math.cos(posicao / (2 ** i))   # posições ímpares: cosseno
        vetor.append(round(valor * 0.3, 2))        # 0.3 = "volume" baixo para não apagar o significado
    return vetor

for p in range(4):
    print("posição", p, "->", vetor_posicao(p, 3))

In [ ]:
def somar_vetores(vetor_a, vetor_b):
    resultado = []
    for i in range(len(vetor_a)):
        resultado.append(round(vetor_a[i] + vetor_b[i], 2))
    return resultado

def frase_para_vetores_com_posicao(frase):
    vetores = []
    for posicao in range(len(frase)):
        embedding = embeddings[frase[posicao]]
        vetores.append(somar_vetores(embedding, vetor_posicao(posicao, 3)))
    return vetores

saidas_1, pesos_1 = atencao(frase_para_vetores_com_posicao(frase_1), W_Q, IDENTIDADE, IDENTIDADE)
saidas_2, pesos_2 = atencao(frase_para_vetores_com_posicao(frase_2), W_Q, IDENTIDADE, IDENTIDADE)

print("'cachorro' na frase 1 (posição 0):", saidas_1[0])
print("'cachorro' na frase 2 (posição 2):", saidas_2[2])

✅ Agora os vetores são **diferentes**! A diferença é pequena porque usamos um "volume" baixo (0.3), mas já é suficiente: durante o treinamento, o modelo aprende a usar essa informação para distinguir **quem mordeu quem**.

## Passo 3 — Múltiplas cabeças de atenção (*multi-head attention*) 🐙

Uma única atenção olha a frase de **um jeito só**. O Transformer usa **várias atenções em paralelo**, cada uma com suas próprias matrizes W_Q, W_K e W_V. Cada uma é chamada de **cabeça**.

É como ler um texto com várias "lentes":
- **Cabeça 1 (gramática):** quem fez a ação? (a nossa W_Q da Parte 3)
- **Cabeça 2 (semelhança):** quais palavras são do mesmo tipo?

No final, os resultados são **colados lado a lado** (concatenados) e passam por mais uma matriz (**W_O**) que volta ao tamanho original.

In [ ]:
W_Q_cabeca1 = W_Q                     # gramática: ação <-> ser vivo/comida
W_Q_cabeca2 = [[3, 0, 0],             # semelhança: procura palavras do mesmo tipo
               [0, 3, 0],
               [0, 0, 3]]

# W_O: junta 6 números (3 de cada cabeça) em 3, fazendo a média das duas cabeças
W_O = [[0.5, 0,   0,   0.5, 0,   0  ],
       [0,   0.5, 0,   0,   0.5, 0  ],
       [0,   0,   0.5, 0,   0,   0.5]]

def multi_head(vetores, mascara=False):
    saidas_1, pesos_1 = atencao(vetores, W_Q_cabeca1, IDENTIDADE, IDENTIDADE, mascara)
    saidas_2, pesos_2 = atencao(vetores, W_Q_cabeca2, IDENTIDADE, IDENTIDADE, mascara)
    resultado = []
    for i in range(len(vetores)):
        concatenado = saidas_1[i] + saidas_2[i]   # + em listas = colar uma na outra
        resultado.append(multiplicar_matriz_vetor(W_O, concatenado))
    return resultado, pesos_1, pesos_2

frase = ["gato", "bebeu", "leite"]
vetores = frase_para_vetores_com_posicao(frase)
saida, pesos_c1, pesos_c2 = multi_head(vetores)

for i in range(len(frase)):
    print("Palavra '" + frase[i] + "'")
    print("  Cabeça 1 (gramática):")
    mostrar_barras(frase, pesos_c1[i])
    print("  Cabeça 2 (semelhança):")
    mostrar_barras(frase, pesos_c2[i])
    print()

🔎 Repare como as duas cabeças prestam atenção em coisas **diferentes**. Os modelos reais têm muitas cabeças (o BERT tem 12 por camada; o GPT-3 tem 96!).

## Passo 4 — Conexão residual, normalização e feed-forward

Depois da atenção, o bloco Transformer faz mais três coisas simples:

| Peça | O que faz | Analogia |
|---|---|---|
| **Conexão residual** | Soma a **entrada** com a **saída** da atenção | Manter o rascunho original e só acrescentar anotações |
| **Normalização** | Coloca os números numa escala parecida (média 0) | Ajustar o volume para nada "estourar" |
| **Feed-forward** | Uma pequena rede neural aplicada a **cada palavra** separadamente | Cada palavra "pensa sozinha" no que aprendeu |

A rede feed-forward usa a função **ReLU**: se o número for negativo, vira 0; se for positivo, continua igual.

In [ ]:
def normalizar(vetor):
    media = 0
    for valor in vetor:
        media = media + valor
    media = media / len(vetor)

    variancia = 0
    for valor in vetor:
        variancia = variancia + (valor - media) ** 2
    variancia = variancia / len(vetor)
    desvio = math.sqrt(variancia + 0.00001)   # + 0.00001 evita divisão por zero

    resultado = []
    for valor in vetor:
        resultado.append(round((valor - media) / desvio, 2))
    return resultado

def relu(vetor):
    resultado = []
    for valor in vetor:
        if valor < 0:
            resultado.append(0)
        else:
            resultado.append(valor)
    return resultado

# Pesos da rede feed-forward (em um modelo real, também são aprendidos)
W_1 = [[1.0, 0.5, 0.0],
       [0.0, 1.0, 0.5],
       [0.5, 0.0, 1.0]]
W_2 = [[0.5, 0.0, 0.5],
       [0.5, 0.5, 0.0],
       [0.0, 0.5, 0.5]]

def feed_forward(vetor):
    camada_1 = relu(multiplicar_matriz_vetor(W_1, vetor))
    return multiplicar_matriz_vetor(W_2, camada_1)

print("normalizar([1, 2, 3])  =", normalizar([1, 2, 3]))
print("relu([-2, 0.5, -0.1])  =", relu([-2, 0.5, -0.1]))
print("feed_forward([1, 0, 1]) =", feed_forward([1, 0, 1]))

## Passo 5 — O bloco Transformer completo 🧱

Agora juntamos tudo. Um **bloco** (ou **camada**) do Transformer faz:

```
entrada ──► Multi-Head Attention ──► (+ entrada) ──► Normalizar ──►
        ──► Feed-Forward ──► (+ o que entrou no feed-forward) ──► Normalizar ──► saída
```

In [ ]:
def bloco_transformer(vetores, mascara=False):
    # 1) atenção com várias cabeças
    saida_atencao, _, _ = multi_head(vetores, mascara)

    # 2) residual + normalização
    depois_atencao = []
    for i in range(len(vetores)):
        depois_atencao.append(normalizar(somar_vetores(vetores[i], saida_atencao[i])))

    # 3) feed-forward + residual + normalização
    saida_final = []
    for vetor in depois_atencao:
        saida_final.append(normalizar(somar_vetores(vetor, feed_forward(vetor))))
    return saida_final

frase = ["o", "gato", "bebeu", "leite"]
vetores = frase_para_vetores_com_posicao(frase)

print("ENTRADA (embedding + posição):")
for i in range(len(frase)):
    print("   ", frase[i].ljust(6), vetores[i])

camada_1 = bloco_transformer(vetores)
print()
print("SAÍDA do bloco 1:")
for i in range(len(frase)):
    print("   ", frase[i].ljust(6), camada_1[i])

camada_2 = bloco_transformer(camada_1)
print()
print("SAÍDA do bloco 2 (empilhado):")
for i in range(len(frase)):
    print("   ", frase[i].ljust(6), camada_2[i])

💡 Como a saída tem o **mesmo tamanho** da entrada, podemos **empilhar** vários blocos. O BERT-base empilha 12; o GPT-3, 96!

## Passo 6 — A máscara: não pode "colar" do futuro 🙈

Modelos que **geram texto** (como o GPT) escrevem uma palavra de cada vez. Durante o treinamento, uma palavra **não pode olhar as palavras que vêm depois** dela — seria como ver a resposta da prova!

Para isso usamos uma **máscara**: os escores das palavras futuras viram `-1000`, e o softmax transforma isso em **0%**.

In [ ]:
frase = ["o", "gato", "bebeu", "leite"]
vetores = frase_para_vetores_com_posicao(frase)

_, pesos_mascara = atencao(vetores, W_Q, IDENTIDADE, IDENTIDADE, mascara=True)

for i in range(len(frase)):
    print("Atenção de '" + frase[i] + "' (só enxerga até ela mesma):")
    mostrar_barras(frase, pesos_mascara[i])
    print()

## Passo 7 — Prevendo a próxima palavra 🔮

Como o ChatGPT escolhe a próxima palavra?

1. Passa a frase pelos blocos Transformer (com máscara);
2. Pega o vetor da **última palavra**;
3. Compara (produto escalar) com o embedding de **cada palavra do vocabulário**;
4. Aplica **softmax** → probabilidade de cada palavra ser a próxima.

⚠️ **Atenção:** nosso modelo **nunca foi treinado** — os números foram escolhidos à mão. Então a previsão pode ser estranha! O **treinamento** é justamente o processo de ajustar milhões (ou bilhões) de números das matrizes até as previsões ficarem boas.

In [ ]:
frase = ["o", "gato", "bebeu"]
vetores = frase_para_vetores_com_posicao(frase)

saida = bloco_transformer(vetores, mascara=True)
saida = bloco_transformer(saida, mascara=True)
ultimo_vetor = saida[len(saida) - 1]

vocabulario = []
escores = []
for palavra in embeddings:
    vocabulario.append(palavra)
    escores.append(produto_escalar(ultimo_vetor, embeddings[palavra]))

probabilidades = softmax(escores)
print("Frase:", " ".join(frase), "___ ?")
print()
mostrar_barras(vocabulario, probabilidades)

🤔 O modelo achou que depois de "o gato bebeu" vem... **outro verbo**! Isso acontece porque ele **não foi treinado**: as matrizes que escolhemos à mão não foram feitas para essa tarefa.

Em um modelo real, o treinamento mostraria milhões de frases como *"o gato bebeu leite"* e ajustaria, aos pouquinhos, todos os números das matrizes (W_Q, W_K, W_V, W_O, W_1, W_2 e os embeddings) até a palavra **"leite"** ganhar a maior probabilidade.

---
## ⭐ Bônus — Um Transformer de verdade em português (opcional)

Agora que você sabe o que acontece por dentro, vamos usar um modelo **já treinado**: o **BERTimbau**, um Transformer treinado com textos em português.

Ele foi treinado para **adivinhar a palavra escondida** (`[MASK]`) usando o contexto dos **dois lados** da frase. Por isso, consegue diferenciar os dois sentidos de "banco"!

> Precisa de internet. O download do modelo (~400 MB) leva cerca de 1 minuto na primeira vez. Podem aparecer **avisos (warnings)** em vermelho durante o carregamento: isso é normal e pode ser ignorado.

In [ ]:
# BONUS - célula opcional
!pip install -q transformers

from transformers import pipeline

preencher = pipeline("fill-mask", model="neuralmind/bert-base-portuguese-cased")

frases = [
    "Fui ao [MASK] sacar dinheiro.",
    "Sentei no [MASK] da praça para descansar.",
    "O gato bebeu o [MASK].",
]

for frase in frases:
    print("Frase:", frase)
    resultados = preencher(frase)
    for r in resultados:
        print("   ", r["token_str"].ljust(12), round(r["score"] * 100, 1), "%")
    print()

---
## ✏️ Exercício final

1. No Passo 1, teste outro par de frases com as mesmas palavras em ordem diferente (ex.: `["gato", "comeu", "queijo"]` e `["queijo", "comeu", "gato"]`), **com e sem** codificação posicional.
2. Crie uma **terceira cabeça** de atenção com uma W_Q inventada por você. Lembre-se de aumentar a `W_O` para 9 colunas!
3. No bônus, escreva **3 frases suas** com `[MASK]` e veja se o modelo acerta.

---
## 📝 Resumo da Parte 4 — O Transformer completo

```
 Texto ─► Tokens ─► Embeddings + Posição
                         │
          ┌──────────────▼──────────────┐
          │  Multi-Head Attention        │  ◄─ Q, K, V, softmax(Q·K/√d)·V
          │  + Residual + Normalização   │
          │  Feed-Forward (ReLU)         │
          │  + Residual + Normalização   │
          └──────────────┬──────────────┘
                         │   (repete N vezes)
                         ▼
          Vetores com CONTEXTO ─► Previsão da próxima palavra
```

| Peça | Para que serve |
|---|---|
| **Codificação posicional** | Informar a ordem das palavras |
| **Multi-head** | Olhar a frase de vários jeitos ao mesmo tempo |
| **Residual + Normalização** | Preservar a informação e manter os números estáveis |
| **Feed-forward** | Processar cada palavra individualmente |
| **Máscara** | Impedir que o modelo veja palavras futuras (GPT) |
| **Treinamento** | Ajustar todos os números das matrizes com muitos textos |

🎓 **Parabéns!** Você construiu um Transformer do zero, só com listas, laços `for` e contas de multiplicar e somar.